# Bangladesh Traffic Violation Detection — Triple-Riding, Helmet & Bangla ANPR

**Seed paper:** Muley, R.R. et al., *"Intelligent Traffic Monitoring System Using Deep Learning: Triple Riding, Automatic License Plate Recognition, and Helmet Detection"*, DoSCI 2024, Springer LNNS vol. 1085, pp. 391–402. DOI: [10.1007/978-981-97-6726-7_31](https://link.springer.com/chapter/10.1007/978-981-97-6726-7_31) · [ResearchGate mirror](https://www.researchgate.net/publication/385582675_Intelligent_Traffic_Monitoring_System_Using_Deep_Learning_Triple_Riding_Automatic_License_Plate_Recognition_and_Helmet_Detection)

The seed paper runs a 4-stage YOLOv5 pipeline (person/motorcycle → helmet → triple-riding → plate) and hands the plate crop to a generic web OCR API for e-challan generation. It's a solid proof of concept but thin on dataset rigor and doesn't address several failure modes documented in a more recent paper on the same problem:

> Deshpande, U.U., Shanbhag, S., Patil, R. et al. (2025). *"Automatic two-wheeler rider identification and triple-riding detection in surveillance systems using deep-learning models."* Discover Artificial Intelligence, 5, 104. DOI: [10.1007/s44163-025-00263-3](https://link.springer.com/article/10.1007/s44163-025-00263-3) (open access)

**What this notebook builds (each stage targets a documented gap):**

| Stage | What it does | Gap it targets |
|---|---|---|
| A | Single YOLOv8n detector: `motorcycle`, `person`, `helmet`, `no_helmet`, `license_plate` | Replaces the seed paper's 3 separate YOLOv5 detectors with one lightweight model that fits a Kaggle T4 |
| B | IoU/proximity-based rider↔motorcycle association | Pedestrians near a bike get miscounted as riders in prior pipelines |
| C | ByteTrack-based multi-frame counting → single/double/triple-riding verdict | Single-frame rectangular-box counting fails under occlusion |
| D | Bangla ANPR on cropped plates (Bangla LPDB-A) | Seed paper used a generic (non-Bangla) web OCR API |
| E | Camera-height/perspective-warp ablation | "Trained near-ground, tested from a traffic pole" is the most-cited failure mode in this literature |
| F (stretch) | Template-based natural-language violation report | Differentiator — no prior paper in this niche does structured captioning instead of raw boxes |

**Before running:** add a Roboflow API key as a Kaggle Secret named `ROBOFLOW_API_KEY` (Add-ons → Secrets), and optionally attach the Kaggle dataset **`andrewmvd/helmet-detection`** via *Add Data* if you want that source too. Written for a Kaggle **T4 x2** notebook.

## Datasets used

| Dataset | Content | Source | Role here |
|---|---|---|---|
| Helmet Detection Project (KMUTT) | 761 imgs — Helmet / License_plate / Motorcyclist, YOLO format | [Roboflow: kmutt-eghvb/helmet-detection-project-rlwya](https://universe.roboflow.com/kmutt-eghvb/helmet-detection-project-rlwya) | Stage A training |
| Triple-riding-detection-fix | 1,067 imgs — Motorcycle / Person / Rider_region | [Roboflow: krokky/triple-riding-detection-fix](https://universe.roboflow.com/krokky/triple-riding-detection-fix) | Stage A training (motorcycle + person classes) |
| Riding (single/double/triple) | 435 imgs labeled by rider count | [Roboflow: ambos-t5pl4/riding](https://universe.roboflow.com/ambos-t5pl4/riding) | Held-out evaluation of the Stage B/C counting logic |
| Bangla LPDB-A | 1,928 vehicle imgs + 2,662 cropped Bangla-script plate imgs, YOLO labels | [Zenodo record 4718238](https://zenodo.org/record/4718238) | Stage D (Bangla ANPR) |
| andrewmvd Helmet Detection (Kaggle) | ~764–1,000 imgs, helmet/no-helmet/plate | Kaggle dataset, add via *Add Data* | Optional extra pretraining/sanity-check data — do **not** treat as a headline benchmark, it's heavily reused already |
| India Driving Dataset (IDD) | 10k+ imgs, unstructured Indian roads, occlusion-heavy | [idd.insaan.iiit.ac.in](https://idd.insaan.iiit.ac.in) | Optional — domain-gap / robustness testing (needs manual registration + download, not scripted below) |

Class counts and exact splits vary per Roboflow *version* — always confirm the version number from the dataset's own "Versions" tab before downloading; the version numbers below are placeholders you should verify.

In [1]:
# Environment setup
!pip install -q ultralytics==8.3.40 roboflow easyocr

import os, glob, shutil, random
from pathlib import Path
from collections import deque, Counter

import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.5/898.5 kB 9.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 60.7 MB/s eta 0:00:0000:0100:01
CUDA available: True
Device: Tesla T4


In [2]:
# Config / paths
try:
    from kaggle_secrets import UserSecretsClient
    ROBOFLOW_API_KEY = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
except Exception:
    ROBOFLOW_API_KEY = "fJCG2f0KsvGm7kHT7Obp"
    # Fallback if you're not using Kaggle Secrets — paste your key below:
    # ROBOFLOW_API_KEY = "your_key_here"

WORK_DIR = Path("/kaggle/working")
DATA_DIR = WORK_DIR / "data_raw"
MERGED_DIR = WORK_DIR / "dataset_merged"
DATA_DIR.mkdir(exist_ok=True, parents=True)
MERGED_DIR.mkdir(exist_ok=True, parents=True)

UNIFIED_CLASSES = ["motorcycle", "person", "helmet", "no_helmet", "license_plate"]


In [ ]:
# Download the Roboflow-hosted datasets
from roboflow import Roboflow

assert ROBOFLOW_API_KEY, "Set ROBOFLOW_API_KEY as a Kaggle Secret (or paste it above) before running this cell."
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

def rf_download(workspace, project, version, out_name, fmt="yolov8"):
    proj = rf.workspace(workspace).project(project)
    try:
        ds = proj.version(version).download(fmt, location=str(DATA_DIR / out_name))
    except RuntimeError as e:
        # Surface which versions actually exist instead of a bare traceback —
        # Roboflow projects can have zero generated versions, in which case
        # no version number will ever work until one is created.
        available = [v.version.split("/")[-1] for v in proj.versions()]
        raise RuntimeError(
            f"{workspace}/{project}: version {version} not found. "
            f"Versions available on this project right now: {available or 'NONE — no dataset version has been generated for this project yet.'} "
            "If none exist, you don't own this public project, so you can't call "
            "generate_version() on it directly — fork it into your own Roboflow "
            "workspace first (or swap in a different, already-versioned dataset)."
        ) from e
    return Path(ds.location)
# ⚠️ Verify each version number on the dataset's Roboflow Universe page before running —
# these are the versions available at the time this notebook was written.
# Using YOUR forked copy of the helmet/plate dataset (the original kmutt-eghvb
# project has no published version, so it can't be downloaded directly — see
# the fork instructions from earlier). Both other sources already had valid
# published versions and don't need forking.
helmet_plate_dir = rf_download("shuvo-t0csp", "helmet-detection-project-rlwya-idvfj", 1, "helmet_plate")
triple_fix_dir   = rf_download("krokky", "triple-riding-detection-fix", 10, "triple_fix")
riding_dir       = rf_download("ambos-t5pl4", "riding", 1, "riding_counts")  # eval-only, see below

print("Downloaded to:")
print(" -", helmet_plate_dir)
print(" -", triple_fix_dir)
print(" -", riding_dir)


In [ ]:
# Optional Kaggle-hosted source + Bangla LPDB-A from Zenodo
ANDREWMVD_DIR = Path("/kaggle/input/helmet-detection")  # adjust to whatever folder name Kaggle assigns after Add Data
if ANDREWMVD_DIR.exists():
    print("andrewmvd helmet dataset attached:", list(ANDREWMVD_DIR.iterdir())[:5])
else:
    print("andrewmvd helmet dataset not attached (optional) — add via Kaggle 'Add Data' if you want this source.")

# Bangla LPDB-A — Zenodo record 4718238 (Shomee & Sams, DICTA 2021)
import requests, zipfile

ZENODO_RECORD = "4718238"
resp = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD}")
resp.raise_for_status()
files = resp.json()["files"]

bangla_lpdb_dir = DATA_DIR / "bangla_lpdb_a"
bangla_lpdb_dir.mkdir(exist_ok=True)

for f in files:
    fname = f["key"]
    url = f["links"]["self"]
    dest = bangla_lpdb_dir / fname
    if dest.exists():
        continue
    print("downloading", fname)
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(dest, "wb") as out:
            for chunk in r.iter_content(chunk_size=1 << 20):
                out.write(chunk)
    if fname.lower().endswith(".zip"):
        with zipfile.ZipFile(dest) as z:
            z.extractall(bangla_lpdb_dir)

print("Bangla LPDB-A ready at:", bangla_lpdb_dir)


In [ ]:
# Remap each source dataset's classes onto the unified schema and merge into one YOLO dataset.
#
# ⚠️ Open each downloaded dataset's data.yaml first and confirm the class *order* — Roboflow
# can reorder classes between export versions, so don't trust the indices below blindly.

CLASS_MAPS = {
    # kmutt-eghvb/helmet-detection-project-rlwya: Helmet, License_plate, Motorcyclist
    "helmet_plate": {0: 2, 1: 4, 2: 1},
    # krokky/triple-riding-detection-fix: Motorcycle, Person, Rider_region
    # (Rider_region is a group box, not a per-instance class — dropped here, index 2 absent)
    "triple_fix": {0: 0, 1: 1},
}

def remap_and_copy(src_dir, class_map, split_ratio=0.85, seed=42):
    src_dir = Path(src_dir)
    img_dir = src_dir / "train" / "images"
    if not img_dir.exists():
        img_dir = src_dir  # fallback for flat exports
    img_paths = sorted(p for p in img_dir.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
    random.seed(seed)
    random.shuffle(img_paths)
    n_train = int(len(img_paths) * split_ratio)

    for i, img_path in enumerate(img_paths):
        split = "train" if i < n_train else "val"
        # Robust to both the standard <split>/images/<split>/labels layout
        # and the flat-export fallback above (where img_dir == src_dir):
        # swap the last "images" path component for "labels" instead of
        # assuming a fixed number of parent hops.
        label_parts = list(img_path.parent.parts)
        label_parts[-1] = "labels" if label_parts[-1] == "images" else label_parts[-1] + "_labels"
        label_path = Path(*label_parts) / (img_path.stem + ".txt")
        (MERGED_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
        (MERGED_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)
        shutil.copy(img_path, MERGED_DIR / "images" / split / img_path.name)

        out_lines = []
        if label_path.exists():
            for line in label_path.read_text().splitlines():
                parts = line.split()
                if not parts:
                    continue
                old_cls = int(parts[0])
                if old_cls not in class_map:
                    continue
                out_lines.append(" ".join([str(class_map[old_cls])] + parts[1:]))
        (MERGED_DIR / "labels" / split / (img_path.stem + ".txt")).write_text("\n".join(out_lines))
    print(f"{src_dir.name}: merged {len(img_paths)} images")

remap_and_copy(helmet_plate_dir, CLASS_MAPS["helmet_plate"])
remap_and_copy(triple_fix_dir, CLASS_MAPS["triple_fix"])

data_yaml_text = f"path: {MERGED_DIR}\ntrain: images/train\nval: images/val\nnames:\n"
for i, c in enumerate(UNIFIED_CLASSES):
    data_yaml_text += f"  {i}: {c}\n"
(MERGED_DIR / "data.yaml").write_text(data_yaml_text)
print(data_yaml_text)


In [ ]:
# Sanity check: draw boxes on a few merged training samples
def draw_yolo_boxes(img_path, label_path, class_names):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if Path(label_path).exists():
        for line in Path(label_path).read_text().splitlines():
            cls, xc, yc, bw, bh = map(float, line.split())
            x1, y1 = int((xc - bw/2) * w), int((yc - bh/2) * h)
            x2, y2 = int((xc + bw/2) * w), int((yc + bh/2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(img, class_names[int(cls)], (x1, max(0, y1 - 5)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)
    return img

sample_imgs = list((MERGED_DIR / "images" / "train").glob("*"))[:4]
fig, axes = plt.subplots(1, len(sample_imgs), figsize=(16, 4))
for ax, img_path in zip(axes, sample_imgs):
    label_path = MERGED_DIR / "labels" / "train" / (img_path.stem + ".txt")
    ax.imshow(draw_yolo_boxes(img_path, label_path, UNIFIED_CLASSES))
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# Stage A — train the unified detector
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # lightweight — fits a single Kaggle T4; try yolov8s.pt if you have more budget

results = model.train(
    data=str(MERGED_DIR / "data.yaml"),
    epochs=60,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    project=str(WORK_DIR / "runs"),
    name="stage_a_detector",
)


In [ ]:
# Validate — headline numbers for the paper's results table
metrics = model.val(data=str(MERGED_DIR / "data.yaml"))
print("mAP50-95:", metrics.box.map)
print("mAP50   :", metrics.box.map50)
print("Per-class mAP50-95:", dict(zip(UNIFIED_CLASSES, metrics.box.maps)))


## Stage B — two-stage rider verification

Directly targets the most-cited false positive in this literature: **pedestrians standing near a bike get counted as riders** because most pipelines trust a single detector's "rider" class with no person↔vehicle association check. This gates every person box against the nearest motorcycle box by IoU/proximity before it counts toward the rider tally.

In [ ]:
def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    a_area = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    b_area = max(0, bx2 - bx1) * max(0, by2 - by1)
    union = a_area + b_area - inter
    return inter / union if union > 0 else 0.0

def box_center(b):
    x1, y1, x2, y2 = b
    return (x1 + x2) / 2, (y1 + y2) / 2

def associate_riders(person_boxes, moto_boxes, proximity_ratio=0.6):
    """Assign each person box to the motorcycle it's plausibly riding.
    A person with no overlapping/nearby motorcycle is dropped (treated as a
    pedestrian), instead of being trusted at face value."""
    assignments = {i: [] for i in range(len(moto_boxes))}
    for p_idx, p in enumerate(person_boxes):
        best_moto, best_score = None, 0.0
        for m_idx, m in enumerate(moto_boxes):
            iou = box_iou(p, m)
            mx1, my1, mx2, my2 = m
            m_w, m_h = mx2 - mx1, my2 - my1
            px, py = box_center(p)
            mx, my = box_center(m)
            dist = ((px - mx) ** 2 + (py - my) ** 2) ** 0.5
            near = dist < proximity_ratio * max(m_w, m_h) * 2
            score = iou if iou > 0 else (0.3 if near else 0.0)
            if score > best_score:
                best_score, best_moto = score, m_idx
        if best_moto is not None and best_score > 0:
            assignments[best_moto].append(p_idx)
    return assignments  # {moto_idx: [person_idx, ...]}


## Stage C — tracking-based rider counting

Single-frame counting breaks under occlusion. This runs ByteTrack (built into `ultralytics`) across a video and takes a majority vote of the rider count over a sliding window per tracked motorcycle, before mapping the count to single/double/triple-riding.

In [ ]:
def classify_riding(count):
    if count <= 1:
        return "single"
    if count == 2:
        return "double"
    return "triple_plus"  # violation

def run_tracking_counts(video_path, model, window=15, moto_cls=0, person_cls=1):
    history = {}
    stream = model.track(source=video_path, tracker="bytetrack.yaml",
                          persist=True, stream=True, classes=[moto_cls, person_cls])
    for r in stream:
        boxes = r.boxes
        if boxes is None or boxes.id is None:
            continue
        cls = boxes.cls.cpu().numpy()
        ids = boxes.id.cpu().numpy()
        xyxy = boxes.xyxy.cpu().numpy()
        motos = [(ids[i], xyxy[i]) for i in range(len(cls)) if cls[i] == moto_cls]
        persons = [xyxy[i] for i in range(len(cls)) if cls[i] == person_cls]
        assign = associate_riders(persons, [m[1] for m in motos])
        for local_idx, (track_id, _) in enumerate(motos):
            count = len(assign.get(local_idx, []))
            history.setdefault(track_id, deque(maxlen=window)).append(count)
    return {tid: classify_riding(Counter(c).most_common(1)[0][0]) for tid, c in history.items()}

# Example — needs a traffic clip attached as a Kaggle input dataset:
# verdicts = run_tracking_counts("/kaggle/input/sample-traffic-clip/clip1.mp4", model)
# print(verdicts)

# Evaluate against the Riding (single/double/triple) held-out set's ground-truth labels
# for the paper's rider-counting accuracy table — load its label file and compare per-image.


## Stage D — Bangla ANPR

The seed paper reads plates with a generic web OCR API. This localizes that stage to Bangla-script plates using Bangla LPDB-A. The stock EasyOCR reader below is a **baseline only** — for the actual paper contribution, fine-tune a recognizer (e.g. a CRNN or PaddleOCR) on Bangla LPDB-A's 2,662 cropped-plate part with its 102-class character annotations, since stock OCR models are not trained on Bangla plate fonts/layouts.

In [ ]:
import easyocr

reader = easyocr.Reader(["bn"], gpu=torch.cuda.is_available())

def read_bangla_plate(plate_crop_bgr):
    result = reader.readtext(plate_crop_bgr)
    return " ".join([r[1] for r in result])

# Baseline sanity check on a few Bangla LPDB-A cropped-plate images
# (point this at the actual "part-2 cropped plates" subfolder once you've inspected the extracted zip)
sample_plates = list(bangla_lpdb_dir.rglob("*.jpg"))[:5] + list(bangla_lpdb_dir.rglob("*.png"))[:5]
for p in sample_plates[:5]:
    img = cv2.imread(str(p))
    if img is not None:
        print(p.name, "->", read_bangla_plate(img))


## Ablation — camera-angle domain gap

Every paper in this space (including both the seed paper and the 2025 follow-up) flags the same failure mode: models trained on near-ground images collapse when tested on real traffic-pole CCTV footage (10–15 ft, steep downward angle). This is a cheap, controlled way to test it: apply a synthetic perspective warp to simulate a higher camera, retrain, and compare against the un-augmented baseline on a genuinely high-angle held-out clip.

In [ ]:
def simulate_camera_height(img, strength=0.3):
    """Perspective-warps a near-ground image toward what a higher,
    steeper-angle traffic-pole camera would see."""
    h, w = img.shape[:2]
    src = np.float32([[0, 0], [w, 0], [0, h], [w, h]])
    shrink = strength * w
    dst = np.float32([[shrink, 0], [w - shrink, 0], [0, h], [w, h]])
    M = cv2.getPerspectiveTransform(src, dst)
    return cv2.warpPerspective(img, M, (w, h))

# Suggested ablation procedure for the paper:
# 1. Duplicate the merged training set; apply simulate_camera_height() at ~30% probability per image
#    (varying `strength` 0.15-0.4 to cover a range of simulated heights).
# 2. Retrain the identical YOLOv8n config on the augmented set.
# 3. Evaluate BOTH the baseline and the augmented model on a held-out clip actually shot from a
#    raised angle (pole/CCTV-style, not near-ground) and report the mAP delta as your headline ablation table.


## Stage F (stretch/novelty) — structured violation report

Instead of raw boxes + OCR text, generate a short natural-language violation record — the same template-conditioned captioning idea used for structured radiology report generation, applied to this domain instead. This is a differentiator reviewers likely haven't seen in this specific niche.

In [ ]:
def generate_violation_report(track_id, riding_verdict, helmet_status, plate_text, confidence):
    violations = []
    if riding_verdict == "triple_plus":
        violations.append("triple riding")
    if helmet_status == "no_helmet":
        violations.append("no helmet")
    if not violations:
        return None
    return (f"Vehicle (track #{track_id}, plate: {plate_text or 'unreadable'}) "
            f"flagged for: {', '.join(violations)}. Confidence: {confidence:.2f}.")

# Example:
# print(generate_violation_report(12, "triple_plus", "no_helmet", "\u09e2\u09e6\u09e6\u09e7", 0.87))


## Paper roadmap

**Suggested results tables:**
1. Stage A detector — per-class mAP50/mAP50-95 (this notebook's `val` cell)
2. Rider-counting accuracy — Stage B+C verdicts vs. ground truth on the Riding (435-img) held-out set
3. Camera-angle ablation — baseline vs. perspective-warp-augmented mAP on a high-angle test clip
4. Bangla ANPR — character/plate-level accuracy, stock EasyOCR vs. fine-tuned recognizer on Bangla LPDB-A

**Open items not scripted here (bigger scope, decide based on your timeline):**
- Full trapezium-box + amodal-regressor reimplementation (Goyal et al., IDD subset) — a heavier lift than the IoU-gating in Stage B, but the more rigorous fix for occlusion-driven false triple-riding counts.
- Real multi-camera/edge deployment benchmarking — every paper in this space flags it as future work; even a single-camera latency/FPS number on a Kaggle T4 would be a novel data point.
- IDD dataset integration for broader occlusion/robustness testing — needs manual registration, not scriptable here.

**Citation list to build your related-work section around:** the seed paper (DoSCI 2024), Deshpande et al. 2025 (Discover AI), Goyal et al. (arXiv:2204.08364, trapezium boxes), and the dataset papers (Bangla LPDB-A / Shomee & Sams, DICTA 2021).